# DyslexiaLens — Multi-Modal, Multi-Task CNN Pipeline
## EMNIST-GAMBO Combined Dataset · Late Fusion · Dual-Head Architecture
---
**Architecture:** Keras Functional API · Custom ACN Layer · Late Fusion  
**Inputs:** Image tensor (128×128×1) + 6 engineered clinical features  
**Outputs:** Classification head (sigmoid) + Severity score head (sigmoid + Huber)  
**Dataset:** EMNIST-GAMBO Balanced (~204k samples) — 1:1 class ratio  
**Baseline:** Test Acc 97.77 % → Operational Acc 97.86 % @ t=0.40 threshold  


## 1. Imports & Environment Setup

In [ ]:
import os, re, zipfile, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.losses import Huber
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")


## 2. Global Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ZIP_EMNIST       = "Dataset_Gambo_EMNIST.zip"
CSV_EMNIST       = "Dataset_Dyslexia_EMNIST_FeatureEngineeringV2.csv"
CSV_NOAUG        = "Dataset_Dyslexia_NoAugmentation_FeatureEngineeringV2.csv"   # kept for reference / ablation
EXTRACT_DIR      = "dataset_extracted"
CHECKPOINT_PATH  = "checkpoints/dyslexialens_best.weights.h5"
os.makedirs("checkpoints", exist_ok=True)

# ── Image config ─────────────────────────────────────────────────────────────
IMG_SIZE         = (128, 128)          # train resolution (source: 224×224)
IMG_CHANNELS     = 1
INPUT_SHAPE_IMG  = (*IMG_SIZE, IMG_CHANNELS)

# ── Feature config ───────────────────────────────────────────────────────────
FEATURE_COLS = [
    "stroke_density",       # replaces legacy 'ink_density' per DS team v2 nomenclature
    "center_of_mass_x",
    "center_of_mass_y",
    "bounding_box_ratio",
    "stroke_transitions",
    "horizontal_symmetry",
]
N_FEATURES       = len(FEATURE_COLS)   # 6

# ── Severity proxy weights ───────────────────────────────────────────────────
# Domain rationale: horizontal_symmetry → letter-reversal signal (dyslexia hallmark)
#                   stroke_transitions  → motor-tremor / hesitation signal
SEV_W_SYMMETRY     = 0.55   # primary: reversal effects
SEV_W_TRANSITIONS  = 0.30   # secondary: motor tremors
SEV_W_BBOX         = 0.15   # tertiary: spatial compression

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE        = 64
EPOCHS            = 50
LEARNING_RATE     = 1e-3
DROPOUT_RATE      = 0.40

# ── Loss weights (multi-task) ────────────────────────────────────────────────
LOSS_WEIGHT_CLS   = 1.0     # classification head priority
LOSS_WEIGHT_SEV   = 0.5     # severity head — fine-tune after clf stability

# ── Threshold sweep ──────────────────────────────────────────────────────────
THRESHOLD_SWEEP_RANGE = np.arange(0.30, 0.71, 0.01)
DEFAULT_THRESHOLD     = 0.40   # proven operational boundary from ablation study

# ── BinaryFocalLoss guard ────────────────────────────────────────────────────
# BLACKLISTED: BinaryFocalLoss destabilises gradients on this handwriting
# structure, collapsing accuracy to ~29 %. Do NOT re-enable.
USE_FOCAL_LOSS = False   # hard guard — never flip this flag
assert not USE_FOCAL_LOSS, "BinaryFocalLoss is blacklisted for DyslexiaLens."

print("Configuration loaded.")
print(f"  Image input shape : {INPUT_SHAPE_IMG}")
print(f"  Feature vector dim: {N_FEATURES}  →  {FEATURE_COLS}")
print(f"  Loss weights      : clf={LOSS_WEIGHT_CLS}, sev={LOSS_WEIGHT_SEV}")
print(f"  Operational thresh: {DEFAULT_THRESHOLD}")


## 3. Dataset Extraction & Inventory

In [ ]:
def extract_zip(zip_path: str, dest: str) -> str:
    """Extract a zip archive to dest, skip if already extracted."""
    dest_path = pathlib.Path(dest)
    if dest_path.exists() and any(dest_path.rglob("*.png")):
        print(f"[SKIP] {zip_path} already extracted → {dest}")
        return str(dest_path)
    print(f"[EXTRACT] {zip_path} → {dest} …")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)
    print(f"[DONE] Extracted {zip_path}")
    return str(dest_path)


# Primary dataset: EMNIST-augmented (balanced 1:1, ~204k)
IMG_ROOT = extract_zip(ZIP_EMNIST, EXTRACT_DIR)

# Inventory
all_imgs = list(pathlib.Path(IMG_ROOT).rglob("*.png"))
print(f"\nTotal .png files found: {len(all_imgs):,}")

# Structure summary
for split in ["Train", "Validation", "Test"]:
    for cat in ["Corrected", "Normal", "Reversal"]:
        count = len(list(pathlib.Path(IMG_ROOT).rglob(f"{split}/{cat}/*.png")))
        print(f"  {split:12s} / {cat:10s} : {count:,}")


## 4. Feature CSV Loading & Severity Proxy Construction

In [ ]:
# ── 4.1  Load primary CSV (EMNIST-balanced) ──────────────────────────────────
df_raw = pd.read_csv(CSV_EMNIST)
print(f"Raw CSV shape    : {df_raw.shape}")
print(f"Class balance    : {df_raw['target_class'].value_counts().to_dict()}")
print(f"Splits           : {df_raw['split'].value_counts().to_dict()}")
print(f"NaN check        : {df_raw.isnull().sum().sum()} total NaNs")


In [ ]:
# ── 4.2  Fix Windows-style image paths to OS-agnostic ────────────────────────
def normalise_path(raw_path: str, img_root: str) -> str:
    """
    Convert Windows backslash paths from the CSV to a resolvable local path.
    
    CSV stores: 'Dataset\\Gambo_EMNIST_Balanced\\Test\\Corrected\\file.png'
    We map the last 3 components (split/category/filename) onto our extraction root.
    """
    parts = re.split(r'[\\/]+', raw_path.strip())
    # Components: [..., split, category, filename]
    rel = os.path.join(*parts[-3:])   # e.g. Test/Corrected/file.png
    return os.path.join(img_root, rel)

df_raw['local_path'] = df_raw['image_path'].apply(
    lambda p: normalise_path(p, IMG_ROOT)
)

# Spot-check first 3
for p in df_raw['local_path'].head(3):
    exists = os.path.exists(p)
    print(f"  {'✓' if exists else '✗'}  {p}")


In [ ]:
# ── 4.3  Severity Proxy Construction ─────────────────────────────────────────
#
# Strategy (agreed specification):
#   Control group (target_class == 0)  → severity = 0.0  (no clinical indicators)
#   Dyslexia group (target_class == 1) → normalized weighted deviation index ∈ [0, 1]
#
# Proxy formula (dyslexia samples only):
#   raw_sev = w_sym  * (1 - horizontal_symmetry)    # asymmetry = reversal signal
#           + w_trans * norm(stroke_transitions)      # motor tremor indicator
#           + w_bbox  * norm(bounding_box_ratio)      # spatial compression
#
# Each sub-component is min-max normalised within the dyslexia sub-population
# before weighting so that each feature contributes on an equal [0,1] scale
# prior to weighting. The composite is then re-normalised to [0,1].

def build_severity_proxy(df: pd.DataFrame) -> pd.Series:
    sev = pd.Series(0.0, index=df.index, dtype=np.float32)

    mask = df['target_class'] == 1
    sub  = df.loc[mask].copy()

    def minmax(s: pd.Series) -> pd.Series:
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo + 1e-8)

    # Reversal component: lower symmetry → higher deviation
    asym  = minmax(1.0 - sub['horizontal_symmetry'])

    # Motor tremor component: more transitions → higher deviation
    trans = minmax(sub['stroke_transitions'])

    # Spatial compression: extreme bbox ratio → higher deviation
    bbox  = minmax(sub['bounding_box_ratio'])

    raw = (
        SEV_W_SYMMETRY    * asym  +
        SEV_W_TRANSITIONS * trans +
        SEV_W_BBOX        * bbox
    )

    # Final normalisation → [0, 1]
    sev.loc[mask] = minmax(raw).values.astype(np.float32)
    return sev


df_raw['severity_proxy'] = build_severity_proxy(df_raw)

print("Severity proxy statistics:")
print(f"  Control  (0) mean : {df_raw.loc[df_raw['target_class']==0,'severity_proxy'].mean():.4f}")
print(f"  Dyslexia (1) mean : {df_raw.loc[df_raw['target_class']==1,'severity_proxy'].mean():.4f}")
print(f"  Dyslexia (1) std  : {df_raw.loc[df_raw['target_class']==1,'severity_proxy'].std():.4f}")
print(f"  Dyslexia (1) range: "
      f"[{df_raw.loc[df_raw['target_class']==1,'severity_proxy'].min():.4f}, "
      f"{df_raw.loc[df_raw['target_class']==1,'severity_proxy'].max():.4f}]")


## 5. Feature Scaling & Dataset Splits

In [ ]:
# ── 5.1  Stratified split using CSV 'split' column (pre-defined) ─────────────
df_train = df_raw[df_raw['split'] == 'Train'].reset_index(drop=True)
df_val   = df_raw[df_raw['split'] == 'Validation'].reset_index(drop=True)
df_test  = df_raw[df_raw['split'] == 'Test'].reset_index(drop=True)

print(f"Train      : {len(df_train):,}  "
      f"(class 0: {(df_train['target_class']==0).sum():,} | "
      f"class 1: {(df_train['target_class']==1).sum():,})")
print(f"Validation : {len(df_val):,}  "
      f"(class 0: {(df_val['target_class']==0).sum():,} | "
      f"class 1: {(df_val['target_class']==1).sum():,})")
print(f"Test       : {len(df_test):,}  "
      f"(class 0: {(df_test['target_class']==0).sum():,} | "
      f"class 1: {(df_test['target_class']==1).sum():,})")


In [ ]:
# ── 5.2  MinMax-scale engineered features (fit ONLY on train) ────────────────
scaler = MinMaxScaler()
scaler.fit(df_train[FEATURE_COLS])

X_feat_train = scaler.transform(df_train[FEATURE_COLS]).astype(np.float32)
X_feat_val   = scaler.transform(df_val[FEATURE_COLS]).astype(np.float32)
X_feat_test  = scaler.transform(df_test[FEATURE_COLS]).astype(np.float32)

y_cls_train  = df_train['target_class'].values.astype(np.float32)
y_cls_val    = df_val['target_class'].values.astype(np.float32)
y_cls_test   = df_test['target_class'].values.astype(np.float32)

y_sev_train  = df_train['severity_proxy'].values.astype(np.float32)
y_sev_val    = df_val['severity_proxy'].values.astype(np.float32)
y_sev_test   = df_test['severity_proxy'].values.astype(np.float32)

print(f"Feature matrix shape — Train: {X_feat_train.shape} | Val: {X_feat_val.shape} | Test: {X_feat_test.shape}")
print(f"Scaler feature range: {scaler.data_min_.round(3)} → {scaler.data_max_.round(3)}")


In [ ]:
# ── 5.3  Class weights for BinaryCrossentropy (balanced training) ────────────
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_cls_train
)
CLASS_WEIGHTS = {0: class_weights_array[0], 1: class_weights_array[1]}
print(f"Class weights: {CLASS_WEIGHTS}")
# NOTE: Multi-output Keras models require class_weight passed as a dict
# keyed by output name. We apply it to the classification head only.
CLS_HEAD_CLASS_WEIGHTS = {
    'classification_head': CLASS_WEIGHTS
}


## 6. tf.data Input Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path: str) -> tf.Tensor:
    """
    Load a PNG from disk, decode to grayscale, resize to 128×128,
    and normalise pixel values to [0, 1].
    
    Source images are stored at 224×224; we downsample to 128×128
    at load time for computational efficiency (agreed spec).
    """
    raw  = tf.io.read_file(path)
    img  = tf.image.decode_png(raw, channels=IMG_CHANNELS)
    img  = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img  = tf.cast(img, tf.float32) / 255.0
    return img


def augment_image(img: tf.Tensor) -> tf.Tensor:
    """
    Lightweight augmentation applied ONLY during training.
    Preserves stroke structure to avoid confounding clinical features.
    """
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.92, upper=1.08)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img


def make_dataset(
    paths, features, y_cls, y_sev,
    batch_size=BATCH_SIZE, training=False, shuffle_buffer=2048
) -> tf.data.Dataset:
    """
    Build a multi-input (image + feature vector), multi-output
    (classification + severity) tf.data.Dataset.
    """
    path_ds   = tf.data.Dataset.from_tensor_slices(paths)
    feat_ds   = tf.data.Dataset.from_tensor_slices(features)
    cls_ds    = tf.data.Dataset.from_tensor_slices(y_cls)
    sev_ds    = tf.data.Dataset.from_tensor_slices(y_sev)

    img_ds = path_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)

    if training:
        img_ds = img_ds.map(augment_image, num_parallel_calls=AUTOTUNE)

    # Zip: inputs = (img, feat) | outputs = {'classification_head': cls, 'severity_head': sev}
    ds = tf.data.Dataset.zip((
        tf.data.Dataset.zip((img_ds, feat_ds)),
        tf.data.Dataset.zip((cls_ds, sev_ds))
    ))

    # Map outputs to named dict so Keras matches them to output layers
    ds = ds.map(
        lambda inputs, targets: (
            {'image_input': inputs[0], 'feature_input': inputs[1]},
            {'classification_head': targets[0], 'severity_head': targets[1]}
        ),
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.shuffle(shuffle_buffer, seed=42, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


# Build all three datasets
train_ds = make_dataset(
    df_train['local_path'].values, X_feat_train, y_cls_train, y_sev_train,
    training=True
)
val_ds = make_dataset(
    df_val['local_path'].values, X_feat_val, y_cls_val, y_sev_val,
    training=False
)
test_ds = make_dataset(
    df_test['local_path'].values, X_feat_test, y_cls_test, y_sev_test,
    training=False
)

# Sanity check — inspect one batch shape
for inputs, targets in train_ds.take(1):
    print("Image input shape  :", inputs['image_input'].shape)
    print("Feature input shape:", inputs['feature_input'].shape)
    print("Classification y   :", targets['classification_head'].shape)
    print("Severity y         :", targets['severity_head'].shape)


## 7. Custom AdaptiveContrastNorm (ACN) Layer

In [ ]:
class AdaptiveContrastNorm(layers.Layer):
    """
    AdaptiveContrastNorm (ACN) — proven custom layer from DyslexiaLens ablation study.
    
    Motivation:
        Handwriting images exhibit highly variable local contrast (ink bleed, paper
        texture, scanner differences). Standard BatchNorm operates on feature maps
        *after* convolution, but ACN acts *on the raw image space* before any
        learnable weights, providing contrast normalisation that is adaptive per
        sample and per spatial region.

    Mechanism:
        For each input sample, ACN computes:
            μ  = mean(x)          — per-sample spatial mean
            σ  = std(x) + ε       — per-sample spatial std (stabilised)
            x̂  = (x − μ) / σ     — zero-mean, unit-variance normalisation
            y  = γ * x̂ + β       — learnable affine re-scaling (2 params per channel)

        γ and β are initialised to 1 and 0 respectively (identity start),
        allowing the network to learn optimal contrast amplification per channel.

    Why NOT BatchNorm here:
        BN computes stats across the batch dimension, making it batch-size
        dependent and unstable at small batch sizes. ACN is fully instance-
        independent, critical for variable-size inference batches at deployment.

    Args:
        epsilon (float): Numerical stability term. Default: 1e-6.
    """

    def __init__(self, epsilon: float = 1e-6, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        # Learnable affine parameters: one gamma and one beta per channel
        channels = input_shape[-1]
        self.gamma = self.add_weight(
            name='gamma',
            shape=(1, 1, 1, channels),
            initializer='ones',
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1, 1, 1, channels),
            initializer='zeros',
            trainable=True
        )
        super().build(input_shape)

    def call(self, x: tf.Tensor, training=None) -> tf.Tensor:
        # Compute per-sample spatial statistics across H and W dimensions
        axes = [1, 2]   # height and width
        mu  = tf.reduce_mean(x, axis=axes, keepdims=True)
        sigma = tf.math.reduce_std(x, axis=axes, keepdims=True) + self.epsilon

        x_norm = (x - mu) / sigma
        return self.gamma * x_norm + self.beta

    def get_config(self):
        config = super().get_config()
        config.update({'epsilon': self.epsilon})
        return config


## 8. Model Architecture — Late Fusion + Dual-Head (Functional API)

In [ ]:
def build_dyslexialens_model(
    img_shape=INPUT_SHAPE_IMG,
    n_features=N_FEATURES,
    dropout_rate=DROPOUT_RATE
) -> Model:
    """
    DyslexiaLens Multi-Modal, Multi-Task Model.

    Architecture Overview:
    ─────────────────────────────────────────────────────────────
    INPUT A (image_input)                  INPUT B (feature_input)
         │                                        │
    AdaptiveContrastNorm (ACN)           Dense(32, relu)
         │                                  BatchNorm
    Conv2D(32, 3×3, relu)                Dense(64, relu)
    BatchNorm → MaxPool → Dropout              │
         │                               feature_embedding
    Conv2D(64, 3×3, relu)
    BatchNorm → MaxPool → Dropout
         │
    Conv2D(128, 3×3, relu)
    BatchNorm → MaxPool → Dropout
         │
    Conv2D(256, 3×3, relu)
    BatchNorm → GlobalAvgPool
         │
    image_embedding
         │
         └──────── Concatenate ────────┘
                        │
                 Dense(256, relu)
                 BatchNorm + Dropout
                 Dense(128, relu)
                 Dropout
                        │
           ┌────────────┴────────────┐
           │                         │
    Dense(1, sigmoid)         Dense(1, sigmoid)
    classification_head       severity_head
    ─────────────────────────────────────────────────────────────
    """

    # ── Input A: Image Branch ─────────────────────────────────────────────────
    image_input = Input(shape=img_shape, name='image_input')

    # ACN: adaptive contrast normalisation (proven ablation winner)
    x = AdaptiveContrastNorm(name='acn')(image_input)

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop1')(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop2')(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    x = layers.Dropout(dropout_rate * 0.5, name='drop3')(x)

    # Block 4 — deeper feature extraction, GlobalAvgPool to avoid spatial overfit
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    image_embedding = layers.GlobalAveragePooling2D(name='image_embedding')(x)

    # ── Input B: Clinical Feature Branch ─────────────────────────────────────
    feature_input = Input(shape=(n_features,), name='feature_input')

    f = layers.Dense(32, activation='relu', name='feat_dense1')(feature_input)
    f = layers.BatchNormalization(name='feat_bn1')(f)
    feature_embedding = layers.Dense(64, activation='relu', name='feature_embedding')(f)

    # ── Late Fusion: Concatenate both embeddings ──────────────────────────────
    fused = layers.Concatenate(name='late_fusion')([image_embedding, feature_embedding])

    # ── Shared Fusion Layers ──────────────────────────────────────────────────
    fused = layers.Dense(256, activation='relu', name='fusion_dense1')(fused)
    fused = layers.BatchNormalization(name='fusion_bn1')(fused)
    fused = layers.Dropout(dropout_rate, name='fusion_drop1')(fused)
    fused = layers.Dense(128, activation='relu', name='fusion_dense2')(fused)
    fused = layers.Dropout(dropout_rate * 0.5, name='fusion_drop2')(fused)

    # ── Head 1: Classification (Dyslexia vs. Control) ─────────────────────────
    # sigmoid → bounded [0,1]; threshold sweep at t=0.40 post-training
    classification_head = layers.Dense(
        1, activation='sigmoid', name='classification_head'
    )(fused)

    # ── Head 2: Severity Score (continuous deviation index) ───────────────────
    # sigmoid → naturally bounded [0,1]; proxy target also in [0,1]
    # Loss: Huber (MAE-aligned, outlier-robust per project rubric)
    severity_head = layers.Dense(
        1, activation='sigmoid', name='severity_head'
    )(fused)

    # ── Assemble Functional Model ─────────────────────────────────────────────
    model = Model(
        inputs=[image_input, feature_input],
        outputs=[classification_head, severity_head],
        name='DyslexiaLens_LateFusion'
    )

    return model


model = build_dyslexialens_model()
model.summary(line_length=100, expand_nested=True)


## 9. Model Compilation

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={
        'classification_head': keras.losses.BinaryCrossentropy(),
        'severity_head':       Huber(delta=0.5),          # MAE-aligned, outlier-robust
    },
    loss_weights={
        'classification_head': LOSS_WEIGHT_CLS,   # 1.0 — classification dominates
        'severity_head':       LOSS_WEIGHT_SEV,   # 0.5 — fine-tuned after clf stable
    },
    metrics={
        'classification_head': [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
        'severity_head': [
            keras.metrics.MeanAbsoluteError(name='mae'),
            keras.metrics.RootMeanSquaredError(name='rmse'),
        ],
    }
)
print("Model compiled successfully.")
print(f"  Optimiser    : Adam (lr={LEARNING_RATE})")
print(f"  Clf loss     : BinaryCrossentropy  (weight={LOSS_WEIGHT_CLS})")
print(f"  Severity loss: Huber(delta=0.5)    (weight={LOSS_WEIGHT_SEV})")
print(f"  [GUARD] BinaryFocalLoss: DISABLED — gradient destabilisation documented")


## 10. Training Callbacks

In [ ]:
callbacks = [
    # Stop if val loss does not improve for 7 epochs; restore best weights
    EarlyStopping(
        monitor='val_classification_head_accuracy',
        patience=7,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    # Halve LR when val accuracy plateaus for 4 epochs
    ReduceLROnPlateau(
        monitor='val_classification_head_accuracy',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        mode='max',
        verbose=1
    ),
    # Save best weights by val classification accuracy
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_classification_head_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max',
        verbose=1
    ),
]
print(f"Callbacks registered:")
for cb in callbacks:
    print(f"  • {cb.__class__.__name__}")


## 11. Model Training

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHTS,   # applied to classification head
    verbose=1
)

print(f"\nTraining complete.")
print(f"  Total epochs run : {len(history.history['loss'])}")
print(f"  Best val clf acc : {max(history.history['val_classification_head_accuracy']):.4f}")
print(f"  Best val sev MAE : {min(history.history['val_severity_head_mae']):.4f}")


## 12. Training History Visualisation

In [ ]:
def plot_training_history(history):
    h = history.history
    epochs_ran = range(1, len(h['loss']) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('DyslexiaLens — Training History', fontsize=15, fontweight='bold')

    # 1. Total loss
    ax = axes[0, 0]
    ax.plot(epochs_ran, h['loss'],     label='Train Loss', color='steelblue')
    ax.plot(epochs_ran, h['val_loss'], label='Val Loss',   color='coral', linestyle='--')
    ax.set_title('Total Combined Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # 2. Classification accuracy
    ax = axes[0, 1]
    ax.plot(epochs_ran, h['classification_head_accuracy'],     label='Train Acc', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_accuracy'], label='Val Acc',   color='coral', linestyle='--')
    ax.set_title('Classification Head — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # 3. Classification AUC
    ax = axes[0, 2]
    ax.plot(epochs_ran, h['classification_head_auc'],     label='Train AUC', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_auc'], label='Val AUC',   color='coral', linestyle='--')
    ax.set_title('Classification Head — AUC-ROC')
    ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
    ax.legend(); ax.grid(alpha=0.3)

    # 4. Severity MAE
    ax = axes[1, 0]
    ax.plot(epochs_ran, h['severity_head_mae'],     label='Train MAE', color='mediumseagreen')
    ax.plot(epochs_ran, h['val_severity_head_mae'], label='Val MAE',   color='orangered', linestyle='--')
    ax.set_title('Severity Head — MAE (Huber-aligned)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MAE')
    ax.legend(); ax.grid(alpha=0.3)

    # 5. Severity RMSE
    ax = axes[1, 1]
    ax.plot(epochs_ran, h['severity_head_rmse'],     label='Train RMSE', color='mediumseagreen')
    ax.plot(epochs_ran, h['val_severity_head_rmse'], label='Val RMSE',   color='orangered', linestyle='--')
    ax.set_title('Severity Head — RMSE')
    ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.legend(); ax.grid(alpha=0.3)

    # 6. Precision / Recall
    ax = axes[1, 2]
    ax.plot(epochs_ran, h['classification_head_precision'],     label='Train Prec', color='steelblue')
    ax.plot(epochs_ran, h['val_classification_head_precision'], label='Val Prec',   color='coral', linestyle='--')
    ax.plot(epochs_ran, h['classification_head_recall'],        label='Train Rec',  color='mediumseagreen')
    ax.plot(epochs_ran, h['val_classification_head_recall'],    label='Val Rec',    color='orangered', linestyle='--')
    ax.set_title('Classification Head — Precision & Recall')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → training_history.png")

plot_training_history(history)


## 13. Post-Training Threshold Sweep (Classification Head)

In [ ]:
# ── 13.1  Collect raw probabilities from validation set ──────────────────────
val_probs_cls = []
val_probs_sev = []
val_true_cls  = []
val_true_sev  = []

for inputs, targets in val_ds:
    p_cls, p_sev = model(inputs, training=False)
    val_probs_cls.append(p_cls.numpy().flatten())
    val_probs_sev.append(p_sev.numpy().flatten())
    val_true_cls.append(targets['classification_head'].numpy().flatten())
    val_true_sev.append(targets['severity_head'].numpy().flatten())

val_probs_cls = np.concatenate(val_probs_cls)
val_probs_sev = np.concatenate(val_probs_sev)
val_true_cls  = np.concatenate(val_true_cls).astype(int)
val_true_sev  = np.concatenate(val_true_sev)

print(f"Validation samples collected: {len(val_probs_cls):,}")


In [ ]:
# ── 13.2  Sweep thresholds from 0.30 → 0.70 ─────────────────────────────────
sweep_results = []
for t in THRESHOLD_SWEEP_RANGE:
    preds = (val_probs_cls >= t).astype(int)
    acc   = np.mean(preds == val_true_cls)
    tp    = np.sum((preds == 1) & (val_true_cls == 1))
    fp    = np.sum((preds == 1) & (val_true_cls == 0))
    fn    = np.sum((preds == 0) & (val_true_cls == 1))
    prec  = tp / (tp + fp + 1e-8)
    rec   = tp / (tp + fn + 1e-8)
    f1    = 2 * prec * rec / (prec + rec + 1e-8)
    sweep_results.append({'threshold': t, 'accuracy': acc, 'precision': prec,
                           'recall': rec, 'f1': f1})

sweep_df = pd.DataFrame(sweep_results)
best_row  = sweep_df.loc[sweep_df['accuracy'].idxmax()]
OPTIMAL_THRESHOLD = best_row['threshold']

print("Threshold Sweep Results:")
print(sweep_df.to_string(index=False, float_format='{:.4f}'.format))
print(f"\n  ► Optimal threshold (max accuracy): t = {OPTIMAL_THRESHOLD:.2f}")
print(f"    Accuracy @ optimal t              : {best_row['accuracy']:.4f}")
print(f"    F1 score @ optimal t              : {best_row['f1']:.4f}")
print(f"    Precision @ optimal t             : {best_row['precision']:.4f}")
print(f"    Recall @ optimal t                : {best_row['recall']:.4f}")
print(f"\n  [BASELINE] Proven t=0.40 → Acc=97.86%, MAE=0.0214")


In [ ]:
# ── 13.3  Visualise threshold sweep ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Sweep — Classification Head', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(sweep_df['threshold'], sweep_df['accuracy'],  label='Accuracy',  color='steelblue')
ax.plot(sweep_df['threshold'], sweep_df['f1'],        label='F1 Score',  color='mediumseagreen')
ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', color='coral')
ax.plot(sweep_df['threshold'], sweep_df['recall'],    label='Recall',    color='orchid')
ax.axvline(OPTIMAL_THRESHOLD, color='black', linestyle='--', label=f't* = {OPTIMAL_THRESHOLD:.2f}')
ax.axvline(DEFAULT_THRESHOLD, color='gray',  linestyle=':',  label=f'Baseline t = {DEFAULT_THRESHOLD:.2f}', alpha=0.7)
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Metrics vs. Decision Threshold')
ax.legend(); ax.grid(alpha=0.3)

# ROC Curve
fpr, tpr, _ = roc_curve(val_true_cls, val_probs_cls)
auc_score   = roc_auc_score(val_true_cls, val_probs_cls)
ax = axes[1]
ax.plot(fpr, tpr, color='steelblue', label=f'ROC (AUC = {auc_score:.4f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Validation Set')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → threshold_sweep.png")


## 14. Final Evaluation on Test Set

In [ ]:
# ── 14.1  Collect test predictions ───────────────────────────────────────────
test_probs_cls = []
test_probs_sev = []
test_true_cls  = []
test_true_sev  = []

for inputs, targets in test_ds:
    p_cls, p_sev = model(inputs, training=False)
    test_probs_cls.append(p_cls.numpy().flatten())
    test_probs_sev.append(p_sev.numpy().flatten())
    test_true_cls.append(targets['classification_head'].numpy().flatten())
    test_true_sev.append(targets['severity_head'].numpy().flatten())

test_probs_cls = np.concatenate(test_probs_cls)
test_probs_sev = np.concatenate(test_probs_sev)
test_true_cls  = np.concatenate(test_true_cls).astype(int)
test_true_sev  = np.concatenate(test_true_sev)

# Apply optimal threshold
test_preds_cls = (test_probs_cls >= OPTIMAL_THRESHOLD).astype(int)

# ── Classification metrics ────────────────────────────────────────────────────
test_acc  = np.mean(test_preds_cls == test_true_cls)
test_auc  = roc_auc_score(test_true_cls, test_probs_cls)

# ── Severity metrics ──────────────────────────────────────────────────────────
test_mae  = np.mean(np.abs(test_probs_sev - test_true_sev))
test_rmse = np.sqrt(np.mean((test_probs_sev - test_true_sev) ** 2))

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)
print(f"  Classification Accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  AUC-ROC                  : {test_auc:.4f}")
print(f"  Severity MAE             : {test_mae:.4f}")
print(f"  Severity RMSE            : {test_rmse:.4f}")
print(f"  Optimal threshold used   : t = {OPTIMAL_THRESHOLD:.2f}")
print("=" * 60)
print()
print("Baseline comparison:")
print(f"  Previous acc @ raw t=0.5 : 97.77%")
print(f"  Previous acc @ t=0.40    : 97.86%  (MAE: 0.0214)")
print()
print(classification_report(
    test_true_cls, test_preds_cls,
    target_names=['Control (0)', 'Dyslexia (1)']
))


In [ ]:
# ── 14.2  Confusion matrix ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DyslexiaLens — Test Set Evaluation', fontsize=13, fontweight='bold')

cm = confusion_matrix(test_true_cls, test_preds_cls)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Control', 'Dyslexia'],
            yticklabels=['Control', 'Dyslexia'])
ax.set_title(f'Confusion Matrix (t = {OPTIMAL_THRESHOLD:.2f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Severity prediction scatter (Dyslexia samples only)
ax = axes[1]
mask_dys = test_true_cls == 1
ax.scatter(test_true_sev[mask_dys], test_probs_sev[mask_dys],
           alpha=0.15, s=8, color='steelblue', label='Dyslexia samples')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.7, label='Perfect prediction')
ax.set_xlabel('True Severity Score')
ax.set_ylabel('Predicted Severity Score')
ax.set_title(f'Severity Head — Prediction vs. Ground Truth\nMAE = {test_mae:.4f}')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → test_evaluation.png")


## 15. Model Persistence

In [ ]:
# Save full model in SavedModel format (preserves custom ACN layer via get_config)
model.save('dyslexialens_model.keras')
print("Full model saved → dyslexialens_model.keras")

# Save feature scaler
import pickle
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Feature scaler saved → feature_scaler.pkl")

# Verify reload with custom ACN layer
reloaded = keras.models.load_model(
    'dyslexialens_model.keras',
    custom_objects={'AdaptiveContrastNorm': AdaptiveContrastNorm}
)
reloaded.summary(line_length=80)
print("\n✓ Model reloaded successfully with custom ACN layer.")


## 16. Inference Utility Function

In [ ]:
def predict_single_sample(
    image_path: str,
    raw_features: dict,
    model=model,
    scaler=scaler,
    threshold: float = OPTIMAL_THRESHOLD
) -> dict:
    """
    Run DyslexiaLens inference on a single handwriting image.

    Args:
        image_path   : Absolute or relative path to a .png image.
        raw_features : Dict with keys matching FEATURE_COLS (unscaled values).
        model        : Trained Keras model.
        scaler       : Fitted MinMaxScaler (from training).
        threshold    : Decision boundary for classification (default: optimal t).

    Returns:
        dict with:
            'dyslexia_probability' : float ∈ [0, 1]
            'predicted_class'      : 0 (Control) or 1 (Dyslexia)
            'label'                : 'Control' or 'Dyslexia'
            'severity_score'       : float ∈ [0, 1] (0 if Control)
            'severity_level'       : 'None' / 'Mild' / 'Moderate' / 'Severe'
    """
    # Load & preprocess image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=IMG_CHANNELS)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.expand_dims(img, 0)   # add batch dim

    # Scale features
    feat_vec = np.array([[raw_features[c] for c in FEATURE_COLS]], dtype=np.float32)
    feat_vec = scaler.transform(feat_vec)
    feat_vec = tf.constant(feat_vec, dtype=tf.float32)

    # Inference
    p_cls, p_sev = model({'image_input': img, 'feature_input': feat_vec}, training=False)
    prob_cls  = float(p_cls.numpy().flatten()[0])
    prob_sev  = float(p_sev.numpy().flatten()[0])

    predicted = int(prob_cls >= threshold)
    label     = 'Dyslexia' if predicted == 1 else 'Control'

    if predicted == 0:
        sev_level = 'None'
    elif prob_sev < 0.33:
        sev_level = 'Mild'
    elif prob_sev < 0.66:
        sev_level = 'Moderate'
    else:
        sev_level = 'Severe'

    return {
        'dyslexia_probability': round(prob_cls, 4),
        'predicted_class':      predicted,
        'label':                label,
        'severity_score':       round(prob_sev if predicted == 1 else 0.0, 4),
        'severity_level':       sev_level,
    }


print("predict_single_sample() ready for deployment integration.")
print()
print("Example call:")
print("""
result = predict_single_sample(
    image_path='path/to/sample.png',
    raw_features={
        'stroke_density':      0.15,
        'center_of_mass_x':    13.2,
        'center_of_mass_y':    13.5,
        'bounding_box_ratio':  1.05,
        'stroke_transitions':  2.14,
        'horizontal_symmetry': 0.83,
    }
)
print(result)
""")
